In [47]:
import importlib
import tests.tokenizer
import os
import json
importlib.reload(tests.tokenizer)
from tests.tokenizer import get_tokenizer
import random

In [36]:
from tests.common import FIXTURES_PATH, gpt2_bytes_to_unicode

VOCAB_PATH = FIXTURES_PATH / "gpt2_vocab.json"
MERGES_PATH = FIXTURES_PATH / "gpt2_merges.txt"

In [20]:
with open("tests/outputs/openweb_vocab.json") as f:
      openweb_vocab = json.load(f)
      openweb_vocab = {int(k):bytes(v) for k,v in openweb_vocab.items()}

In [21]:
with open("tests/outputs/openweb_merges.json") as f:
      openweb_merges = json.load(f)
      openweb_merges = [(bytes(a), bytes(b)) for (a,b) in openweb_merges]

In [53]:
with open("tests/outputs/tiny_stories_vocab.json") as f:
      tinystories_vocab = json.load(f)
      tinystories_vocab = {int(k):bytes(v) for k,v in tinystories_vocab.items()}


with open("tests/outputs/tiny_stories_merges.json") as f:
  tinystories_merges = json.load(f)
  tinystories_merges = [(bytes(a), bytes(b)) for (a,b) in tinystories_merges]

tinystories_tokenizer = get_tokenizer(tinystories_vocab, tinystories_merges, ["<|endoftext|>"])

In [26]:
openweb_tokenizer = get_tokenizer(openweb_vocab, openweb_merges, ["<|endoftext|>"])

In [61]:
all_ids = []
stop = 10000
ind = 0
with open("tests/data/owt_.txt") as f:
    for _id in openweb_tokenizer.encode_iterable(f):
        all_ids.append(_id)
        ind += 1
        if ind % 1e6 == 0:
            print(ind)

            
arr = np.array(all_ids, dtype=np.uint16)
arr.tofile("tokens.bin")

# read back:
arr = np.fromfile("tokens.bin", dtype=np.uint16)


1000000
2000000
3000000
4000000
5000000
6000000
7000000
8000000
9000000
10000000
11000000
12000000
13000000
14000000
15000000
16000000
17000000
18000000
19000000
20000000
21000000
22000000
23000000
24000000
25000000
26000000
27000000
28000000
29000000
30000000
31000000
32000000
33000000
34000000
35000000
36000000
37000000
38000000
39000000
40000000
41000000
42000000
43000000
44000000
45000000
46000000
47000000
48000000
49000000
50000000
51000000
52000000
53000000
54000000
55000000
56000000
57000000
58000000
59000000
60000000
61000000
62000000
63000000
64000000
65000000
66000000
67000000
68000000
69000000
70000000
71000000
72000000
73000000
74000000
75000000
76000000
77000000
78000000
79000000
80000000
81000000
82000000
83000000
84000000
85000000
86000000
87000000
88000000
89000000
90000000
91000000
92000000
93000000
94000000
95000000
96000000
97000000
98000000
99000000
100000000
101000000
102000000
103000000
104000000
105000000
106000000
107000000
108000000
109000000
110000000
11100000

In [64]:
!pip install tqdm

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /usr/local/fbcode/platform010/Python3.12.framework/Versions/3.12/bin/python3.12 -m pip install --upgrade pip


In [66]:
import numpy as np
from tqdm import tqdm


def tokenize_to_file(input_path, output_path, tokenizer, dtype=np.uint16):
  all_ids = []
  total_bytes = os.path.getsize(input_path)

  with open(input_path) as f, tqdm(
      total=total_bytes, unit="B", unit_scale=True, desc=input_path
  ) as pbar:
      def lines_with_progress():
          for line in f:
              pbar.update(len(line.encode("utf-8")))
              yield line

      for _id in tokenizer.encode_iterable(lines_with_progress()):
          all_ids.append(_id)

  arr = np.array(all_ids, dtype=dtype)
  arr.tofile(output_path)
  print(f"{input_path} -> {output_path}: {len(arr):,} tokens")
  return arr

files = [
  ("tests/data/owt_valid.txt",                "owt_valid.bin",         openweb_tokenizer),
  ("tests/data/TinyStoriesV2-GPT4-train.txt", "tinystories_train.bin", tinystories_tokenizer),
  ("tests/data/TinyStoriesV2-GPT4-valid.txt", "tinystories_valid.bin", tinystories_tokenizer),
]

for input_path, output_path, tok in files:
  tokenize_to_file(input_path, output_path, tok)

tests/data/owt_valid.txt: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 290M/290M [03:54<00:00, 1.24MB/s]


tests/data/owt_valid.txt -> owt_valid.bin: 66,636,800 tokens


tests/data/TinyStoriesV2-GPT4-train.txt: 100%|███████████████████████████████████████████████████████████████████████████| 2.23G/2.23G [25:44<00:00, 1.44MB/s]


tests/data/TinyStoriesV2-GPT4-train.txt -> tinystories_train.bin: 542,162,443 tokens


tests/data/TinyStoriesV2-GPT4-valid.txt: 100%|███████████████████████████████████████████████████████████████████████████| 22.5M/22.5M [00:15<00:00, 1.45MB/s]


tests/data/TinyStoriesV2-GPT4-valid.txt -> tinystories_valid.bin: 5,469,063 tokens


In [62]:
import numpy as np

# uint16 if vocab ≤ 65535, else uint32
arr = np.array(all_ids, dtype=np.uint16)
arr.tofile("tokens.bin")

# read back:
arr = np.fromfile("tokens.bin", dtype=np.uint16)

In [46]:
with open("tests/data/owt_train.txt") as f:
      text = f.read(10_000_000)
      splited = text.split("<|endoftext|>")

print(len(splited))

2055


In [51]:
random_docs = random.sample(splited, 10)
tot_bytes = 0
tot_tokens = 0

for s in random_docs:
    tot_bytes += len(s.encode("utf-8"))
    tot_tokens += len(openweb_tokenizer.encode(s))

print(tot_bytes, tot_tokens, f"{tot_bytes / tot_tokens:.2f} bytes/token")

50515 10942 4.62 bytes/token


 Sample 10 documents from TinyStories and OpenWebText. Using your previously-trained
TinyStories and OpenWebText tokenizers (10K and 32K vocabulary size, respectively),
encode these sampled documents into integer IDs. What is each tokenizer’s compression ratio
(bytes/token)?
4.68bytes/token


In [55]:
tot_bytes = 0
tot_tokens = 0
for s in random_docs:
    tot_bytes += len(s.encode("utf-8"))
    tot_tokens += len(tinystories_tokenizer.encode(s))

print(tot_bytes, tot_tokens, f"{tot_bytes / tot_tokens:.2f} bytes/token")

50515 15243 3.31 bytes/token


What happens if you tokenize your OpenWebText sample with the TinyStories tokenizer?
Compare the compression ratio and/or qualitatively describe what happens. 

Tinystories compression ratio is worse

In [ ]:
Estimate the throughput of your tokenizer (e.g., in bytes/second). How long would it take to
tokenize the Pile dataset (825GB of text)?

In [56]:
import time

tot_bytes = 0
tot_tokens = 0
start = time.perf_counter()

for i, s in enumerate(splited):
  if i >= 1000:
      break
  tot_bytes += len(s.encode("utf-8"))
  tot_tokens += len(openweb_tokenizer.encode(s))

elapsed = time.perf_counter() - start
print(f"{tot_bytes} bytes, {tot_tokens} tokens, {elapsed:.2f}s")
print(f"{tot_bytes / elapsed:,.0f} bytes/s, {tot_bytes / tot_tokens:.2f} bytes/token")

4815850 bytes, 1097439 tokens, 3.65s
1,318,624 bytes/s, 4.39 bytes/token


1.26MB per second. 852GB takes 852000/1.26=  676190 seconds = 187 hours

In [60]:
with open("tests/data/owt_train.txt") as f:
      text = f.read()
      train_splited = text.split("<|endoftext|>")
    
with open("tests/data/owt_valid.txt") as f:
      text = f.read()
      eval_splited = text.split("<|endoftext|>")

KeyboardInterrupt: 

In [57]:
852000/1.26

676190.4761904762

In [58]:
676190/3600

187.83055555555555

In [158]:
tokenizer.encode_word(" ate") 

[10, 3]

In [159]:
word_bytes = "cat".encode("utf-8")
[b for b in word_bytes]

[99, 97, 116]

In [31]:
def get_tokenizer_from_vocab_merges_path(
    vocab_path: str | os.PathLike,
    merges_path: str | os.PathLike,
    special_tokens: list[str] | None = None,
):
    gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
    with open(vocab_path) as vocab_f:
        gpt2_vocab = json.load(vocab_f)
    gpt2_bpe_merges = []
    with open(merges_path) as f:
        for line in f:
            cleaned_line = line.rstrip()
            if cleaned_line and len(cleaned_line.split(" ")) == 2:
                gpt2_bpe_merges.append(tuple(cleaned_line.split(" ")))
    # The GPT-2 tokenizer uses a remapped unicode encoding for bytes. Let's
    # just return the original bytes, so we don't force students to use
    # any particular encoding scheme.
    vocab = {
        gpt2_vocab_index: bytes([gpt2_byte_decoder[token] for token in gpt2_vocab_item])
        for gpt2_vocab_item, gpt2_vocab_index in gpt2_vocab.items()
    }
    # If any of the special tokens don't exist in the vocab, append them to the vocab.
    if special_tokens:
        for special_token in special_tokens:
            byte_encoded_special_token = special_token.encode("utf-8")
            if byte_encoded_special_token not in set(vocab.values()):
                vocab[len(vocab)] = byte_encoded_special_token

    merges = [
        (
            bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
            bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
        )
        for merge_token_1, merge_token_2 in gpt2_bpe_merges
    ]
    return get_tokenizer(vocab, merges, special_tokens)



In [32]:
from tests.common import FIXTURES_PATH, gpt2_bytes_to_unicode

VOCAB_PATH = FIXTURES_PATH / "gpt2_vocab.json"
MERGES_PATH = FIXTURES_PATH / "gpt2_merges.txt"

In [33]:
def test_encode_iterable_tinystories_sample_roundtrip():
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH,
        merges_path=MERGES_PATH,
    )
    all_ids = []
    with open(FIXTURES_PATH / "tinystories_sample.txt") as f:
        for _id in tokenizer.encode_iterable(f):
            all_ids.append(_id)
    with open(FIXTURES_PATH / "tinystories_sample.txt") as f:
        corpus_contents = f.read()
    assert tokenizer.decode(all_ids) == corpus_contents

In [34]:
sorted(["<|endoftext|>", "<|endoftext|><|endoftext|>"]).reverse()

In [35]:
test_encode_iterable_tinystories_sample_roundtrip()